# 第32章 饼图与环形图（pie）

在类别很少且总和具有明确整体含义时使用饼图或环形图表达占比。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。学习重点不是“把图画出来”，而是让图表服务于一个可回答的问题。


## 适用场景

展示2至5个互斥类别在同一整体中的比例。

## 数据结构

一列非负数值，类别互斥且总和代表完整整体。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 startangle 从 90 改为 0，观察扇区起始位置变化
2. 修改 autopct 格式从 "%.1f%%" 为 "%.0f%%"，对比百分比精度显示
3. 调整 wedgeprops 中的 width 参数（如 0.5 或 0.3），说明环形宽度对中心空间的影响


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from js import window
base_url = window.location.origin
transactions = pd.read_csv(f"{base_url}/datasets/uci_online_retail_200k.csv", parse_dates=["InvoiceDate"])
transactions["amount"] = transactions["Quantity"] * transactions["UnitPrice"]
transactions["month"] = transactions["InvoiceDate"].dt.to_period("M").astype("string")
completed = transactions.query("Quantity > 0 and UnitPrice > 0")
monthly_summary = completed.groupby("month").agg(sales=("amount", "sum"), orders=("InvoiceNo", "nunique"))
months = monthly_summary.index.to_numpy()
sales = (monthly_summary["sales"] / 10_000).to_numpy()
orders = monthly_summary["orders"].to_numpy()
profit = sales * 0.18
top_countries = completed.groupby("Country")["amount"].sum().nlargest(4).index
country_rows = transactions[transactions["Country"].isin(top_countries)].copy()
country_rows["flow"] = np.where(country_rows["Quantity"] > 0, "销售", "退货")
country_rows["amount_abs"] = country_rows["amount"].abs()
regional_summary = country_rows.pivot_table(index="Country", columns="flow", values="amount_abs", aggfunc="sum", fill_value=0) / 10_000
regions = regional_summary.index.to_numpy()
online = regional_summary.get("销售", pd.Series(0, index=regional_summary.index)).to_numpy()
offline = regional_summary.get("退货", pd.Series(0, index=regional_summary.index)).to_numpy()
samples = completed["amount"].sample(2_000, random_state=25).to_numpy()
print(f"UCI Online Retail：{len(transactions):,} 行；图表使用聚合结果与固定样本")


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
channel_sales = np.array([180, 92, 58])
labels = ["自然流量", "广告", "会员"]
fig, ax = plt.subplots(figsize=(6.5, 5))
ax.pie(channel_sales, labels=labels, autopct="%.1f%%", startangle=90, colors=["#1a73e8", "#f9ab00", "#188038"], wedgeprops={"edgecolor": "white"})
ax.set_title("销售渠道占比")
fig.tight_layout()
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
wedges, texts, autotexts = ax.pie(channel_sales, labels=labels, autopct="%.1f%%", startangle=90, pctdistance=0.78, colors=["#1a73e8", "#f9ab00", "#188038"], wedgeprops={"width": 0.38, "edgecolor": "white"})
ax.text(0, 0, f"总计\n{channel_sales.sum()}", ha="center", va="center", fontsize=14, fontweight="bold")
ax.set_title("销售渠道构成（环形图）")
fig.tight_layout()
plt.show()


## 3. 参数说明

- autopct：百分比
- startangle：起始角
- wedgeprops：扇区样式
- explode：轻微突出


## 4. 结果解读

主要读取最大、最小和大致占比；精确比较仍应使用柱状图。


## 常见误区

- 类别过多
- 使用3D效果
- 多个饼图之间比较角度
- 数据并非同一整体


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


In [ ]:
satisfaction = np.array([72, 20, 8])
fig, ax = plt.subplots(figsize=(6.5, 5))
ax.pie(satisfaction, labels=["满意", "一般", "不满意"], autopct="%1.0f%%", startangle=90, colors=["#188038", "#f9ab00", "#d93025"], wedgeprops={"width": 0.42, "edgecolor": "white"})
ax.set_title("客户满意度构成")
fig.tight_layout()
plt.show()


## 本章小结

在类别很少且总和具有明确整体含义时使用饼图或环形图表达占比。


### 你已经掌握

- 判断饼图与环形图（pie）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 展示2至5个互斥类别在同一整体中的比例。 |
| 数据结构 | 一列非负数值，类别互斥且总和代表完整整体。 |
| 结果解读 | 主要读取最大、最小和大致占比；精确比较仍应使用柱状图。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `autopct` | 百分比 |
| `startangle` | 起始角 |
| `wedgeprops` | 扇区样式 |
| `explode` | 轻微突出 |


### 需要注意

- 类别过多
- 使用3D效果
- 多个饼图之间比较角度
- 数据并非同一整体


### 完成检查

- [ ] 能判断什么问题适合使用饼图与环形图（pie）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
